# Using the Panoptes Aggregation Tool from Zooniverse
See Readme for the installation instructions.


This notebook combines the aggregation steps from the raw classification from zooniverse.
Installed from github, https://aggregation-caesar.zooniverse.org/Scripts.html#scripts
The latest pypi release is from 2020 and I can't install it: https://pypi.org/project/panoptes-aggregation/

When using the aggregation tool, the following steps are necessary:
- change input_path and output_path to your local paths. Those will be necessary later i.e. for plots.
- adobt "subjects_path" for the subjects file and "annotations_source" for the classification file in config.py
- later in the section Panoptes Data Extraction change the paths acordingly
- change the paths to the classification files in the panoptes extraction sections. At first this is in the config file "config.py"


In [1]:
# install the aggregation tool
!pip install -U git+https://github.com/zooniverse/aggregation-for-caesar.git

# this worked with panoptes-aggregation version 4.1.0

!pip show panoptes-aggregation

  Cloning https://github.com/zooniverse/aggregation-for-caesar.git to /private/var/folders/2k/78nn7s4548986wsjh29rhj9w0000gn/T/pip-req-build-0fjt8p32
  Running command git clone --filter=blob:none --quiet https://github.com/zooniverse/aggregation-for-caesar.git /private/var/folders/2k/78nn7s4548986wsjh29rhj9w0000gn/T/pip-req-build-0fjt8p32
  Resolved https://github.com/zooniverse/aggregation-for-caesar.git to commit c8a9fffb5228b962515eecd96e852ae0aebaa681
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Name: panoptes_aggregation
Version: 4.1.0
Summary: Aggregation code for Zooniverse panoptes projects.
Home-page: 
Author: 
Author-email: Coleman Krawczyk <coleman@zooniverse.org>
License: 
Location: /Users/christian/opt/anaconda3/envs/iguanas-from-above-zooniverse_310/lib/python3.10/site-packages
Requires: beautifulsoup4, collatex, hdbscan, lxml, numpy, packa

In [2]:
# !pip uninstall -y panoptes_aggregation

In [3]:
## This does not work because they are not updating their packages anymore
# !pip install panoptes_aggregation

In [4]:
!pip install -r requirements-dev.txt

In [5]:
from zooniverse.config import get_config
import pandas as pd
from pathlib import Path

### use either the subset of the subset
#phase_tag = "Iguanas 1st launch"
#data_folder = "./data/phase_1"
## 

# phase_tag = "Iguanas 2nd launch"
# data_folder = "./data/phase_2"

# phase_tag = "Iguanas 3rd launch"
# data_folder = "./data/phase_3"

phase_tag = "Iguanas 4th launch"
data_folder = "./data/phase_4"

# 
input_path = Path("/Users/christian/data/zooniverse")
# use_gold_standard_subset = "expert" # Use the expert-GS-Xphase as the basis


output_path = Path("/Users/christian/data/zooniverse/2024_09_25_analysis").joinpath(phase_tag).resolve()

workflow_id_p1 = 14370.0
workflow_id_p2 = 20600.0
workflow_id_p3 = 22040.0
workflow_id_p4 = 25351.0

output_plot_path = output_path.joinpath("plots")
output_plot_path.mkdir(parents=True, exist_ok=True)

reprocess = False

config = get_config(phase_tag=phase_tag, input_path=input_path, output_path=output_path)
config


{'annotations_source': PosixPath('/Users/christian/data/zooniverse/IguanasFromAbove/2024-08-08/iguanas-from-above-classifications.csv'),
 'goldstandard_data': PosixPath('/Users/christian/data/zooniverse/Images/Zooniverse_Goldstandard_images/expert-GS-4thphase_renamed.csv'),
 'gold_standard_image_subset': PosixPath('/Users/christian/data/zooniverse/Images/Zooniverse_Goldstandard_images/4-T2-GS-results-5th-0s.csv'),
 'image_source': None,
 'subjects_path': PosixPath('/Users/christian/data/zooniverse/IguanasFromAbove/2024-08-08/iguanas-from-above-subjects.csv'),
 'yes_no_dataset': PosixPath('/Users/christian/data/zooniverse/2024_09_25_analysis/Iguanas 4th launch/yes_no_dataset_Iguanas 4th launch.csv'),
 'flat_dataset': PosixPath('/Users/christian/data/zooniverse/2024_09_25_analysis/Iguanas 4th launch/flat_dataset_Iguanas 4th launch.csv'),
 'flat_panoptes_points': PosixPath('/Users/christian/data/zooniverse/2024_09_25_analysis/Iguanas 4th launch/flat_panoptes_points_Iguanas 4th launch.csv'

# Look into the subjects file
This contains the mappings from the subject_id to the image file

In [6]:
# read the original subjects file
df_subjects = pd.read_csv(config["subjects_path"], sep=",")

# filter the subjects for only the images in the three phases

df_subjects = df_subjects[df_subjects.workflow_id.isin([workflow_id_p1, workflow_id_p2, workflow_id_p3, workflow_id_p4])]

# inspect the metadata
import json
def get_json_keys(json_str):
    try:
        json_obj = json.loads(json_str)
        return list(json_obj.keys())
    except json.JSONDecodeError:
        return []

# Apply the function to each row in the metadata column and collect all keys
all_keys = df_subjects['locations'].apply(get_json_keys)

# Flatten the list of lists and get unique keys
unique_keys = set([key for sublist in all_keys for key in sublist])

print(unique_keys)

{'0'}


Clean up the subjects file for inconsistent naming.

In [7]:
df_subjects["image_name"] = df_subjects['metadata'].apply(lambda x: json.loads(x).get('Image_name') 
                                        or json.loads(x).get('image_name') 
                                        or json.loads(x).get('Filename')).sort_values(ascending=True)

# 'site', 'flight', 'Flight', 'Site', 'flight_code' depict the same
df_subjects["flight_code"] = df_subjects['metadata'].apply(lambda x: json.loads(x).get('flight_code') 
                                        or json.loads(x).get('site') 
                                        or json.loads(x).get('flight')
                                        or json.loads(x).get('Flight')
                                        or json.loads(x).get('Site')).sort_values(ascending=True)

df_subjects["url"] = df_subjects['locations'].apply(lambda x: json.loads(x)["0"])
df_subjects["filepath"] = None

In [8]:
df_subjects

,subject_id,project_id,workflow_id,subject_set_id,metadata,locations,classifications_count,retired_at,retirement_reason,created_at,updated_at,image_name,flight_code,url,filepath
190,47967468,11905,14370.0,86008,"{""site"":""SFB"",""image_name"":""SFB01-3_08.jpg"",""s...","{""0"":""https://panoptes-uploads.zooniverse.org/...",20,2020-11-15 19:06:16 UTC,classification_count,2020-07-18 20:38:14 UTC,2020-07-18 20:38:14 UTC,SFB01-3_08.jpg,SFB,https://panoptes-uploads.zooniverse.org/subjec...,None
191,47967469,11905,14370.0,86008,"{""site"":""SFB"",""image_name"":""SFB01-3_15.jpg"",""s...","{""0"":""https://panoptes-uploads.zooniverse.org/...",20,2020-10-28 19:25:18 UTC,classification_count,2020-07-18 20:38:17 UTC,2020-07-18 20:38:17 UTC,SFB01-3_15.jpg,SFB,https://panoptes-uploads.zooniverse.org/subjec...,None
192,47967470,11905,14370.0,86008,"{""site"":""SFB"",""image_name"":""SFB01-3_27.jpg"",""s...","{""0"":""https://panoptes-uploads.zooniverse.org/...",20,2020-11-14 10:07:19 UTC,classification_count,2020-07-18 20:38:18 UTC,2020-07-18 20:38:18 UTC,SFB01-3_27.jpg,SFB,https://panoptes-uploads.zooniverse.org/subjec...,None
193,47967471,11905,14370.0,86008,"{""site"":""SFB"",""image_name"":""SFB01-3_28.jpg"",""s...","{""0"":""https://panoptes-uploads.zooniverse.org/...",20,2020-11-09 10:36:02 UTC,classification_count,2020-07-18 20:38:20 UTC,2020-07-18 20:38:20 UTC,SFB01-3_28.jpg,SFB,https://panoptes-uploads.zooniverse.org/subjec...,None
194,47967472,11905,14370.0,86008,"{""site"":""SFB"",""image_name"":""SFB01-3_34.jpg"",""s...","{""0"":""https://panoptes-uploads.zooniverse.org/...",20,2020-11-18 20:44:36 UTC,classification_count,2020-07-18 20:38:22 UTC,2020-07-18 20:38:22 UTC,SFB01-3_34.jpg,SFB,https://panoptes-uploads.zooniverse.org/subjec...,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68155,92469731,11905,25351.0,116619,"{""Site"":""Puerto Villamil"",""Island"":""Isabela"",""...","{""0"":""https://panoptes-uploads.zooniverse.org/...",20,2024-05-22 05:12:27 UTC,classification_count,2023-10-12 16:28:45 UTC,2023-10-12 16:28:45 UTC,Isa_ISVB02_27012023_155.jpg,Puerto Villamil,https://panoptes-uploads.zooniverse.org/subjec...,None
68156,92469733,11905,25351.0,116619,"{""Site"":""Puerto Villamil"",""Island"":""Isabela"",""...","{""0"":""https://panoptes-uploads.zooniverse.org/...",19,2024-04-13 19:24:15 UTC,classification_count,2023-10-12 16:28:46 UTC,2023-10-12 16:28:46 UTC,Isa_ISVB01_27012023_75_36.jpg,Puerto Villamil,https://panoptes-uploads.zooniverse.org/subjec...,None
68157,92469734,11905,25351.0,116619,"{""Site"":""Puerto Villamil"",""Island"":""Isabela"",""...","{""0"":""https://panoptes-uploads.zooniverse.org/...",20,2024-04-21 08:49:10 UTC,classification_count,2023-10-12 16:28:46 UTC,2023-10-12 16:28:46 UTC,Isa_ISVB01_27012023_27_16.jpg,Puerto Villamil,https://panoptes-uploads.zooniverse.org/subjec...,None
68158,92469735,11905,25351.0,116619,"{""Site"":""Puerto Villamil"",""Island"":""Isabela"",""...","{""0"":""https://panoptes-uploads.zooniverse.org/...",20,2024-06-26 17:32:21 UTC,classification_count,2023-10-12 16:28:46 UTC,2023-10-12 16:28:46 UTC,Isa_ISVB03_77012023_28_17.jpg,Puerto Villamil,https://panoptes-uploads.zooniverse.org/subjec...,None


helper function to download the images using the urls in the subjects file

In [9]:
from loguru import logger
from time import sleep

import requests

def download_image(url, filename):
    """Download an image from a URL and save it to a file."""
    try:
        response = requests.get(url)
        if response.status_code == 200:
            with open(filename, 'wb') as file:
                file.write(response.content)
            return True
        else:
            logger.warning(f"Failed to download {url}")
            logger.error(response)
            sleep(5)
            return False
    except Exception as e:
        logger.error(e)
        sleep(5)
        return False

# Panoptes Data Extraction from Zooniverse
## Panoptes config
### Create the configuration files automatically
The configurations were changed to custom workflow versions.

In [10]:
# create a configuration file from the workflow
#!mkdir ./data/phase_1
#! panoptes_aggregation config /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-workflows.csv 14370 --min_version 0 --max_version 142.245 -d ./data/phase_1
# 
#!mkdir ./data/phase_2
#! panoptes_aggregation config /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-workflows.csv 20600 --min_version 0 --max_version 94.166 -d ./data/phase_2
# 
#!mkdir ./data/phase_3
#! panoptes_aggregation config /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-workflows.csv 22040 --min_version 0 --max_version 9.63 -d ./data/phase_3

# !mkdir ./data/phase_4
# ! panoptes_aggregation config /Users/christian/data/zooniverse/IguanasFromAbove/2024-08-08/iguanas-from-above-workflows.csv 25351 -v 51.133 -d ./data/phase_4


## Have a look at the datasets

In [11]:
!tail /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv | grep workflow_translation_id

516322421,jenniferccguy,2484629,5df3221c2868fbefce6d,25379,Plastics GS dataset,15.25,2023-10-15 18:36:12 UTC,,,"{""source"":""api"",""session"":""8327675098ea28ef4ed9f73d92d97a6af828eae69368ee3880c555e7ddfb0b10"",""viewport"":{""width"":1440,""height"":783},""started_at"":""2023-10-15T18:36:02.119Z"",""user_agent"":""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36"",""utc_offset"":""-3600"",""finished_at"":""2023-10-15T18:36:12.785Z"",""live_project"":true,""interventions"":{""opt_in"":true,""messageShown"":false},""user_language"":""en"",""user_group_ids"":[],""subject_dimensions"":[{""clientWidth"":720,""clientHeight"":705,""naturalWidth"":1053,""naturalHeight"":1030}],""subject_selection_state"":{""retired"":false,""selected_at"":""2023-10-15T18:35:11.684Z"",""already_seen"":false,""selection_state"":""normal"",""finished_workflow"":false,""user_has_finished_workflow"":false},""workflow_translation_id"":""68195""}

In [12]:
!tail /Users/christian/data/zooniverse/IguanasFromAbove/2024-08-08/iguanas-from-above-classifications.csv

570348141,Ekrouse,2169311,dc443c2a3eb398451a07,25351,Iguanas 4th launch,51.133,2024-06-30 19:32:11 UTC,,,"{""source"":""api"",""session"":""e7afd21d7e2fd8b501f25dfef7aaa452335325c66153fb356d495f73849b4904"",""viewport"":{""width"":1138,""height"":640},""started_at"":""2024-06-29T23:15:41.205Z"",""user_agent"":""Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"",""utc_offset"":""14400"",""finished_at"":""2024-06-30T19:32:11.454Z"",""live_project"":true,""interventions"":{""opt_in"":true,""messageShown"":false},""user_language"":""en"",""user_group_ids"":[],""subject_dimensions"":[{""clientWidth"":580,""clientHeight"":576,""naturalWidth"":719,""naturalHeight"":714}],""subject_selection_state"":{""retired"":false,""selected_at"":""2024-06-29T23:15:13.814Z"",""already_seen"":false,""selection_state"":""normal"",""finished_workflow"":false,""user_has_finished_workflow"":false},""workflow_translation_id"":""68096""}","[{""task"":""T0"",""t

## Extract the data

In [13]:
# phase 1
if data_folder == "./data/phase_1":
    !mkdir ./data/phase_1/V121.144
    !mkdir ./data/phase_1/V134.236
    
    !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv -d ./data/phase_1/V121.144 ./data/phase_1/Extractor_config_workflow_14370_V121.144.yaml
    
    !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv -d ./data/phase_1/V134.236 ./data/phase_1/Extractor_config_workflow_14370_V134.236-1.yaml
    
else:
    print(f"No data to process because of the data_folder: {data_folder}")


No data to process because of the data_folder: ./data/phase_4


In [14]:
if data_folder == "./data/phase_2":
    # phase 2
    
    !mkdir ./data/phase_2/V89.162
    !mkdir ./data/phase_2/V93.166
    !mkdir ./data/phase_2/V94.166 
    
    !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv -d ./data/phase_2/V89.162 ./data/phase_2/Extractor_config_workflow_20600_V89.162.yaml
    !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv -d ./data/phase_2/V93.166 ./data/phase_2/Extractor_config_workflow_20600_V93.166.yaml
    !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv -d ./data/phase_2/V94.166 ./data/phase_2/Extractor_config_workflow_20600_V94.166.yaml

else:
    print(f"No data to process because of the data_folder: {data_folder}")

No data to process because of the data_folder: ./data/phase_4


## Extacting data based on the classifications from 2023-10-15


In [15]:
if data_folder == "./data/phase_3":
    !mkdir ./data/phase_3/V7.63    
    !mkdir ./data/phase_3/V9.63

    !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv -d ./data/phase_3/V7.63 ./data/phase_3/Extractor_config_workflow_22040_V7.63.yaml
    !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv -d ./data/phase_3/V9.63 ./data/phase_3/Extractor_config_workflow_22040_V9.63.yaml

else:
    print(f"No data to process because of the data_folder: {data_folder}")

No data to process because of the data_folder: ./data/phase_4


## Extacting data based on the newer classifications from 2024-08-08

In [16]:
if data_folder == "./data/phase_3":
    !mkdir ./data/phase_3/V7.63_08-08
    !mkdir ./data/phase_3/V9.63_08-08

    !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2024-08-08/iguanas-from-above-classifications.csv -d ./data/phase_3/V7.63_08-08 ./data/phase_3/Extractor_config_workflow_22040_V7.63.yaml
    !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2024-08-08/iguanas-from-above-classifications.csv -d ./data/phase_3/V9.63_08-08 ./data/phase_3/Extractor_config_workflow_22040_V9.63.yaml

else:
    print(f"No data to process because of the data_folder: {data_folder}")

No data to process because of the data_folder: ./data/phase_4


In [17]:
if data_folder == "./data/phase_4":
    !mkdir ./data/phase_4/V51.133    

    !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2024-08-08/iguanas-from-above-classifications.csv -d ./data/phase_4/V51.133 ./data/phase_4/Extractor_config_workflow_25351_V51.133.yaml


else:
    print(f"No data to process because of the data_folder: {data_folder}")

mkdir: ./data/phase_4/V51.133: File exists
/Users/christian/opt/anaconda3/envs/iguanas-from-above-zooniverse_310/lib/python3.10/site-packages/panoptes_aggregation/scripts/extract_panoptes_csv.py:68: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  classifications = pandas.read_csv(classification_csv_in, encoding='utf-8', dtype={'workflow_version': str})
Extracting: 100% |#############################################| Time:  0:00:49


### Merge the single point and questions extractions

In [18]:
# phase 1
if data_folder == "./data/phase_1":
    df_panoptes_point_extractor_1 = pd.read_csv(f"./data/phase_1/V121.144/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_2 = pd.read_csv(f"./data/phase_1/V134.236/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_1["workflow_version"] = "121.144"
    df_panoptes_point_extractor_2["workflow_version"] = "134.236"
    
    df_panoptes_question_1 = pd.read_csv(f"{data_folder}/V121.144/question_extractor_extractions.csv", sep=",")
    df_panoptes_question_2 = pd.read_csv(f"{data_folder}/V134.236/question_extractor_extractions.csv", sep=",")
    
    df_panoptes_point_extractor = pd.concat([df_panoptes_point_extractor_1, df_panoptes_point_extractor_2], axis=0)
    df_panoptes_question = pd.concat([df_panoptes_question_1, df_panoptes_question_2], axis=0)
    
    df_panoptes_point_extractor
    
else:
    print(f"No data to process because of the data_folder: {data_folder}")

No data to process because of the data_folder: ./data/phase_4


In [19]:
# # phase 2
if data_folder == "./data/phase_2":
    # read the rectangles annotations too there
    df_panotes_rectangle_extractor_1 = pd.read_csv(f"{data_folder}/V89.162/shape_extractor_rectangle_extractions.csv", sep=",")
    
    df_panoptes_point_extractor_1 = pd.read_csv(f"{data_folder}/V89.162/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_2 = pd.read_csv(f"{data_folder}/V93.166/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_3 = pd.read_csv(f"{data_folder}/V94.166/point_extractor_by_frame_extractions.csv", sep=",")
    
    df_panoptes_point_extractor_1["workflow_version"] = "89.162"
    df_panoptes_point_extractor_2["workflow_version"] = "93.166"
    df_panoptes_point_extractor_3["workflow_version"] = "94.166"
    
    df_panoptes_question_1 = pd.read_csv(f"{data_folder}/V89.162/question_extractor_extractions.csv", sep=",")
    df_panoptes_question_2 = pd.read_csv(f"{data_folder}/V93.166/question_extractor_extractions.csv", sep=",")
    df_panoptes_question_3 = pd.read_csv(f"{data_folder}/V94.166/question_extractor_extractions.csv", sep=",")
    
    df_panoptes_point_extractor = pd.concat([df_panoptes_point_extractor_1, df_panoptes_point_extractor_2, df_panoptes_point_extractor_3], axis=0)
    df_panoptes_question = pd.concat([df_panoptes_question_1, df_panoptes_question_2, df_panoptes_question_3], axis=0)

    df_panotes_rectangle_extractor_1
    
else:
    print(f"No data to process because of the data_folder: {data_folder}")

No data to process because of the data_folder: ./data/phase_4


### phase 3 - 2023 data

In [20]:
if data_folder == "./data/phase_3":
    df_panoptes_point_extractor_1 = pd.read_csv(f"{data_folder}/V7.63/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_2 = pd.read_csv(f"{data_folder}/V9.63/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_1["workflow_version"] = "7.63"
    df_panoptes_point_extractor_2["workflow_version"] = "9.63"

    df_panoptes_question_1 = pd.read_csv(f"{data_folder}/V7.63/question_extractor_extractions.csv", sep=",")
    df_panoptes_question_2 = pd.read_csv(f"{data_folder}/V9.63/question_extractor_extractions.csv", sep=",")
    df_panoptes_question_1["workflow_version"] = "7.63"
    df_panoptes_question_2["workflow_version"] = "9.63"

    df_panoptes_point_extractor = pd.concat([df_panoptes_point_extractor_1, df_panoptes_point_extractor_2], axis=0)
    df_panoptes_question = pd.concat([df_panoptes_question_1, df_panoptes_question_2], axis=0)
    
else:
    print(f"No data to process because of the data_folder: {data_folder}")



No data to process because of the data_folder: ./data/phase_4


NameError: name 'df_panoptes_point_extractor' is not defined

### phase 3 from  08-08-2024

In [22]:
if data_folder == "./data/phase_3":
    df_panoptes_point_extractor_1 = pd.read_csv(f"{data_folder}/V7.63_08-08/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_2 = pd.read_csv(f"{data_folder}/V9.63_08-08/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_1["workflow_version"] = "7.63"
    df_panoptes_point_extractor_2["workflow_version"] = "9.63"

    df_panoptes_question_1 = pd.read_csv(f"{data_folder}/V7.63_08-08/question_extractor_extractions.csv", sep=",")
    df_panoptes_question_2 = pd.read_csv(f"{data_folder}/V7.63_08-08/question_extractor_extractions.csv", sep=",")
    df_panoptes_question_1["workflow_version"] = "7.63"
    df_panoptes_question_2["workflow_version"] = "9.63"

    df_panoptes_point_extractor = pd.concat([df_panoptes_point_extractor_1, df_panoptes_point_extractor_2], axis=0)
    df_panoptes_question = pd.concat([df_panoptes_question_1, df_panoptes_question_2], axis=0)
    
else:
    print(f"No data to process because of the data_folder: {data_folder}")

No data to process because of the data_folder: ./data/phase_4


## Phase 4

In [23]:
# phase 4
if data_folder == "./data/phase_4":
    df_panoptes_point_extractor_1 = pd.read_csv(f"{data_folder}/V51.133/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_1["workflow_version"] = "51.133"

    df_panoptes_question_1 = pd.read_csv(f"{data_folder}/V51.133/question_extractor_extractions.csv", sep=",")
    df_panoptes_question_1["workflow_version"] = "51.133"

    df_panoptes_point_extractor = pd.concat([df_panoptes_point_extractor_1], axis=0)
    df_panoptes_question = pd.concat([df_panoptes_question_1], axis=0)
    
else:
    print(f"No data to process because of the data_folder: {data_folder}")

In [24]:
df_panoptes_point_extractor.drop(columns=["user_name", "user_id"], inplace=False)

,classification_id,workflow_id,task,created_at,subject_id,extractor,data.aggregation_version,data.frame0.T2_tool2_x,data.frame0.T2_tool2_y,data.frame0.T4_tool1_x,...,data.frame0.T4_tool6_y,data.frame0.T4_tool8_x,data.frame0.T4_tool8_y,data.frame0.T2_tool3_x,data.frame0.T2_tool3_y,data.frame0.T2_tool1_x,data.frame0.T2_tool1_y,data.frame0.T4_tool3_x,data.frame0.T4_tool3_y,workflow_version
0,516391010,25351,T2,2023-10-16 10:10:44 UTC,92462758,point_extractor_by_frame,4.1.0,[509.775390625],[664.4249267578125],NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
1,516391010,25351,T4,2023-10-16 10:10:44 UTC,92462758,point_extractor_by_frame,4.1.0,NaN,NaN,[470.683349609375],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
2,516403493,25351,T2,2023-10-16 12:34:09 UTC,92464036,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
3,516403493,25351,T4,2023-10-16 12:34:09 UTC,92464036,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
4,516403739,25351,T2,2023-10-16 12:36:25 UTC,92464442,point_extractor_by_frame,4.1.0,"[571.8541870117188, 679.4479370117188, 591.519...","[126.17318725585938, 71.54948425292969, 22.937...",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404635,570813276,25351,T4,2024-07-03 09:38:38 UTC,92461607,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
404636,570813416,25351,T2,2024-07-03 09:40:06 UTC,92458660,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
404637,570813416,25351,T4,2024-07-03 09:40:06 UTC,92458660,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
404638,570813530,25351,T2,2024-07-03 09:41:28 UTC,92460485,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133


In [25]:
df_panoptes_point_extractor.sort_values(by=[ "created_at"], ascending=False)

,classification_id,user_name,user_id,workflow_id,task,created_at,subject_id,extractor,data.aggregation_version,data.frame0.T2_tool2_x,...,data.frame0.T4_tool6_y,data.frame0.T4_tool8_x,data.frame0.T4_tool8_y,data.frame0.T2_tool3_x,data.frame0.T2_tool3_y,data.frame0.T2_tool1_x,data.frame0.T2_tool1_y,data.frame0.T4_tool3_x,data.frame0.T4_tool3_y,workflow_version
404639,570813530,Ramalina,2606938.0,25351,T4,2024-07-03 09:41:28 UTC,92460485,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
404638,570813530,Ramalina,2606938.0,25351,T2,2024-07-03 09:41:28 UTC,92460485,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
404637,570813416,Ramalina,2606938.0,25351,T4,2024-07-03 09:40:06 UTC,92458660,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
404636,570813416,Ramalina,2606938.0,25351,T2,2024-07-03 09:40:06 UTC,92458660,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
404635,570813276,Ramalina,2606938.0,25351,T4,2024-07-03 09:38:38 UTC,92461607,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5,516403739,ANDREAVARELA89,1983945.0,25351,T4,2023-10-16 12:36:25 UTC,92464442,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
2,516403493,ANDREAVARELA89,1983945.0,25351,T2,2023-10-16 12:34:09 UTC,92464036,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
3,516403493,ANDREAVARELA89,1983945.0,25351,T4,2023-10-16 12:34:09 UTC,92464036,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133
1,516391010,ANDREAVARELA89,1983945.0,25351,T4,2023-10-16 10:10:44 UTC,92462758,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133


## Merge the subjects file with the point extractor file to get the image name
This is necessary to get the image name for the points.

In [26]:
# join the image name from the subjects file
df_panoptes_point_extractor = df_panoptes_point_extractor.merge(df_subjects[["subject_id", "image_name"]], left_on="subject_id", right_on="subject_id")
df_panoptes_point_extractor = df_panoptes_point_extractor[df_panoptes_point_extractor.subject_id.isin(df_subjects.subject_id)]

df_panoptes_point_extractor

,classification_id,user_name,user_id,workflow_id,task,created_at,subject_id,extractor,data.aggregation_version,data.frame0.T2_tool2_x,...,data.frame0.T4_tool8_x,data.frame0.T4_tool8_y,data.frame0.T2_tool3_x,data.frame0.T2_tool3_y,data.frame0.T2_tool1_x,data.frame0.T2_tool1_y,data.frame0.T4_tool3_x,data.frame0.T4_tool3_y,workflow_version,image_name
0,516391010,ANDREAVARELA89,1983945.0,25351,T2,2023-10-16 10:10:44 UTC,92462758,point_extractor_by_frame,4.1.0,[509.775390625],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133,Fer_FPE03-04-05_18122021_118_14.jpg
1,516391010,ANDREAVARELA89,1983945.0,25351,T4,2023-10-16 10:10:44 UTC,92462758,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133,Fer_FPE03-04-05_18122021_118_14.jpg
2,520603679,Kanesh-Gandhi,2644392.0,25351,T2,2023-11-05 18:25:57 UTC,92462758,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133,Fer_FPE03-04-05_18122021_118_14.jpg
3,520603679,Kanesh-Gandhi,2644392.0,25351,T4,2023-11-05 18:25:57 UTC,92462758,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133,Fer_FPE03-04-05_18122021_118_14.jpg
4,523311597,katilling,2678358.0,25351,T2,2023-11-18 02:33:13 UTC,92462758,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,[508.81048583984375],[665.7697143554688],NaN,NaN,51.133,Fer_FPE03-04-05_18122021_118_14.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404635,569258394,thermomole,2701687.0,25351,T4,2024-06-24 12:16:33 UTC,92469395,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133,Isa_ISVB01_27012023_57_29.jpg
404636,569377745,BirdLeaf,1734449.0,25351,T2,2024-06-24 23:40:00 UTC,92469395,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133,Isa_ISVB01_27012023_57_29.jpg
404637,569377745,BirdLeaf,1734449.0,25351,T4,2024-06-24 23:40:00 UTC,92469395,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133,Isa_ISVB01_27012023_57_29.jpg
404638,569672478,Siddharth1,1836105.0,25351,T2,2024-06-26 18:47:01 UTC,92469395,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133,Isa_ISVB01_27012023_57_29.jpg


## Anonymise the data

In [27]:
from hashlib import blake2b

df_panoptes_point_extractor["user_id"] = df_panoptes_point_extractor['user_id'].apply(lambda x: blake2b(str(x).encode(), digest_size=16).hexdigest() if not pd.isnull(x) else x)
# Anonymize 'user_name' by hashing
df_panoptes_point_extractor['user_name'] = df_panoptes_point_extractor['user_name'].apply(lambda x: blake2b(x.encode(), digest_size=16).hexdigest() if isinstance(x, str) else x)

df_panoptes_question["user_id"] = df_panoptes_question['user_id'].apply(lambda x: blake2b(str(x).encode(), digest_size=16).hexdigest() if not pd.isnull(x) else x)
# Anonymize 'user_name' by hashing
df_panoptes_question['user_name'] = df_panoptes_question['user_name'].apply(lambda x: blake2b(x.encode(), digest_size=16).hexdigest() if isinstance(x, str) else x)

In [28]:
df_panoptes_question[df_panoptes_question["data.yes"] == 1.0].groupby("subject_id").size().sort_values(ascending=False)

subject_id
92462183    38
92463214    28
92459074    26
92463918    24
92458601    24
            ..
92464574     1
92460502     1
92464578     1
92460495     1
92459960     1
Length: 6145, dtype: int64

## Determine the amount of yes Answers for "Is there an Iguana"

In [29]:
df_panoptes_question

,classification_id,user_name,user_id,workflow_id,task,created_at,subject_id,extractor,data.yes,data.aggregation_version,data.no,workflow_version
0,516391010,386fc0ec047b7e259744e72e8e64b9f9,ea57b1088a10fa7fef30ed0b344e2ca3,25351,T0,2023-10-16 10:10:44 UTC,92462758,question_extractor,1.0,4.1.0,NaN,51.133
1,516403493,386fc0ec047b7e259744e72e8e64b9f9,ea57b1088a10fa7fef30ed0b344e2ca3,25351,T0,2023-10-16 12:34:09 UTC,92464036,question_extractor,NaN,4.1.0,1.0,51.133
2,516403739,386fc0ec047b7e259744e72e8e64b9f9,ea57b1088a10fa7fef30ed0b344e2ca3,25351,T0,2023-10-16 12:36:25 UTC,92464442,question_extractor,1.0,4.1.0,NaN,51.133
3,516405060,d2618fb74db893e8bb165cb30a48416d,NaN,25351,T0,2023-10-16 12:49:03 UTC,92462615,question_extractor,NaN,4.1.0,1.0,51.133
4,516405305,0718143716f54f03e5311b0d801830ef,NaN,25351,T0,2023-10-16 12:51:11 UTC,92459041,question_extractor,1.0,4.1.0,NaN,51.133
...,...,...,...,...,...,...,...,...,...,...,...,...
202315,570813037,1243c157ab8b76cc35166a4fe97a6f91,676a88d0e03d10c46bdea529074f09f0,25351,T0,2024-07-03 09:36:00 UTC,92460149,question_extractor,NaN,4.1.0,1.0,51.133
202316,570813145,1243c157ab8b76cc35166a4fe97a6f91,676a88d0e03d10c46bdea529074f09f0,25351,T0,2024-07-03 09:36:57 UTC,92463653,question_extractor,NaN,4.1.0,1.0,51.133
202317,570813276,1243c157ab8b76cc35166a4fe97a6f91,676a88d0e03d10c46bdea529074f09f0,25351,T0,2024-07-03 09:38:38 UTC,92461607,question_extractor,NaN,4.1.0,1.0,51.133
202318,570813416,1243c157ab8b76cc35166a4fe97a6f91,676a88d0e03d10c46bdea529074f09f0,25351,T0,2024-07-03 09:40:06 UTC,92458660,question_extractor,NaN,4.1.0,1.0,51.133


In [30]:
df_panoptes_question_r = df_panoptes_question[df_panoptes_question.task == "T0"][["subject_id", "data.no", "data.yes"]].groupby("subject_id").sum()

df_panoptes_question_r = df_panoptes_question_r.reset_index()
df_panoptes_question_r = df_panoptes_question_r[df_panoptes_question_r.subject_id.isin(df_subjects.subject_id)]
df_panoptes_question_r

,subject_id,data.no,data.yes
0,92458387,0.0,22.0
1,92458388,4.0,16.0
2,92458389,3.0,17.0
3,92458390,1.0,19.0
4,92458391,21.0,0.0
...,...,...,...
9923,92469731,9.0,11.0
9924,92469733,0.0,20.0
9925,92469734,1.0,19.0
9926,92469735,20.0,0.0


In [31]:
df_panoptes_question_r.to_csv(output_path / config["panoptes_question"], index = False)

## Get the Point Marks Analysis Ready

Filter for T2 only

In [32]:
df_panoptes_point_extractor_r = df_panoptes_point_extractor[
    (df_panoptes_point_extractor.task == "T2")
]
df_panoptes_point_extractor_r.columns

Index(['classification_id', 'user_name', 'user_id', 'workflow_id', 'task',
       'created_at', 'subject_id', 'extractor', 'data.aggregation_version',
       'data.frame0.T2_tool2_x', 'data.frame0.T2_tool2_y',
       'data.frame0.T4_tool1_x', 'data.frame0.T4_tool1_y',
       'data.frame0.T4_tool4_x', 'data.frame0.T4_tool4_y',
       'data.frame0.T4_tool0_x', 'data.frame0.T4_tool0_y',
       'data.frame0.T4_tool2_x', 'data.frame0.T4_tool2_y',
       'data.frame0.T2_tool0_x', 'data.frame0.T2_tool0_y',
       'data.frame0.T4_tool5_x', 'data.frame0.T4_tool5_y',
       'data.frame0.T4_tool7_x', 'data.frame0.T4_tool7_y',
       'data.frame0.T4_tool6_x', 'data.frame0.T4_tool6_y',
       'data.frame0.T4_tool8_x', 'data.frame0.T4_tool8_y',
       'data.frame0.T2_tool3_x', 'data.frame0.T2_tool3_y',
       'data.frame0.T2_tool1_x', 'data.frame0.T2_tool1_y',
       'data.frame0.T4_tool3_x', 'data.frame0.T4_tool3_y', 'workflow_version',
       'image_name'],
      dtype='object')

### Which tool is which now?
| Tool Name               | Classification                               |
|-------------------------|----------------------------------------------|
| data.frame0.T2_tool0_x  | Adult Male in a lek                          |
| data.frame0.T2_tool1_x  | Adult Male alone                             |
| data.frame0.T2_tool2_x  | Others (females, young males, juveniles)     |
| data.frame0.T2_tool3_x  | Partial iguana                               |
| data.frame0.T2_tool4_x  | Could be an iguana, not sure                 |

Is "Could be an iguana, not sure" and "Partial Iguana" are omitted.


In [33]:
df_panoptes_point_extractor_r

,classification_id,user_name,user_id,workflow_id,task,created_at,subject_id,extractor,data.aggregation_version,data.frame0.T2_tool2_x,...,data.frame0.T4_tool8_x,data.frame0.T4_tool8_y,data.frame0.T2_tool3_x,data.frame0.T2_tool3_y,data.frame0.T2_tool1_x,data.frame0.T2_tool1_y,data.frame0.T4_tool3_x,data.frame0.T4_tool3_y,workflow_version,image_name
0,516391010,386fc0ec047b7e259744e72e8e64b9f9,ea57b1088a10fa7fef30ed0b344e2ca3,25351,T2,2023-10-16 10:10:44 UTC,92462758,point_extractor_by_frame,4.1.0,[509.775390625],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133,Fer_FPE03-04-05_18122021_118_14.jpg
2,520603679,3e5c95da2f84e6f04e971408dbdab8d7,1ed243d8863f93f280b5aecce0c2f80c,25351,T2,2023-11-05 18:25:57 UTC,92462758,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133,Fer_FPE03-04-05_18122021_118_14.jpg
4,523311597,83dd7d12ba62f69e40d9bd8b5eeb23b0,dab35ca4005258caceaab79f37e23712,25351,T2,2023-11-18 02:33:13 UTC,92462758,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,[508.81048583984375],[665.7697143554688],NaN,NaN,51.133,Fer_FPE03-04-05_18122021_118_14.jpg
6,528076882,5b7f6cd059096be78a76988c60f1c741,NaN,25351,T2,2023-12-09 19:12:49 UTC,92462758,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,[509.3125],[663.75],NaN,NaN,51.133,Fer_FPE03-04-05_18122021_118_14.jpg
8,529018098,386fa01a49140278f5494d0042b9f20a,27a60a267e4e0f1d976cac30038e8fe3,25351,T2,2023-12-14 04:32:06 UTC,92462758,point_extractor_by_frame,4.1.0,[509.70989990234375],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133,Fer_FPE03-04-05_18122021_118_14.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404630,566921121,dce9e8bb730153bf0d0bad3f52b853e0,b0e10539db360fb487870c2b5dff8aaa,25351,T2,2024-06-11 15:39:30 UTC,92469395,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133,Isa_ISVB01_27012023_57_29.jpg
404632,567651844,940c519bb3437ec73a4fc6071eae5041,8e606b31ff94fd9d6ba2151bb5d1d030,25351,T2,2024-06-15 09:53:39 UTC,92469395,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133,Isa_ISVB01_27012023_57_29.jpg
404634,569258394,ba42128d23c5d1e18111062e13f8c2d3,00b4196b5d562fbd25d0585353c627f9,25351,T2,2024-06-24 12:16:33 UTC,92469395,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133,Isa_ISVB01_27012023_57_29.jpg
404636,569377745,17d2b0a53f67a0ccba57a9b0aec20f50,92c6091eaf50a7f1019c6cd81f8dd5e4,25351,T2,2024-06-24 23:40:00 UTC,92469395,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.133,Isa_ISVB01_27012023_57_29.jpg


In [34]:
# create a flat structure from the nested marks over multiple columns from that.
from ast import literal_eval

expected_columns = {'data.frame0.T2_tool0_x', 'data.frame0.T2_tool1_x', 'data.frame0.T2_tool2_x', 'data.frame0.T2_tool0_y', 'data.frame0.T2_tool1_y', 'data.frame0.T2_tool2_y'}
assert expected_columns.issubset(df_panoptes_point_extractor_r.columns), f"Missing columns: {expected_columns - set(df_panoptes_point_extractor_r.columns)}" 

# In phase 3 Tool 4 does not exist
if data_folder == "./data/phase_3" or data_folder == "./data/phase_4": 
    columns_keep_x = ['data.frame0.T2_tool0_x', 'data.frame0.T2_tool1_x', 'data.frame0.T2_tool2_x']
    columns_keep_y = ['data.frame0.T2_tool0_y', 'data.frame0.T2_tool1_y', 'data.frame0.T2_tool2_y']


else:
    expected_columns = expected_columns + {'data.frame0.T2_tool4_x', 'data.frame0.T2_tool4_y'}
    assert expected_columns.issubset(df_panoptes_point_extractor_r.columns), f"Missing columns: {expected_columns - set(df_panoptes_point_extractor_r.columns)}" 
    columns_keep_x = ['data.frame0.T2_tool0_x', 'data.frame0.T2_tool1_x', 'data.frame0.T2_tool2_x', 'data.frame0.T2_tool4_x']
    columns_keep_y = ['data.frame0.T2_tool0_y', 'data.frame0.T2_tool1_y', 'data.frame0.T2_tool2_y', 'data.frame0.T2_tool4_y']

for col in columns_keep_x + columns_keep_y:
    df_panoptes_point_extractor_r[col] = df_panoptes_point_extractor_r[col].apply(lambda x: literal_eval(x) if pd.notnull(x) else [])

# Merge the lists in 'x' and 'y' coordinates
df_panoptes_point_extractor_r['x'] = df_panoptes_point_extractor_r[columns_keep_x].values.tolist()
df_panoptes_point_extractor_r['y'] = df_panoptes_point_extractor_r[columns_keep_y].values.tolist()

# Flatten the lists in each row for 'x' and 'y'
df_panoptes_point_extractor_r['x'] = df_panoptes_point_extractor_r['x'].apply(lambda x: [item for sublist in x for item in sublist])
df_panoptes_point_extractor_r['y'] = df_panoptes_point_extractor_r['y'].apply(lambda x: [item for sublist in x for item in sublist])

# Explode the DataFrame to separate rows for each x, y pair
# Explode the DataFrame based on these columns to get separate rows for each list element
df_panoptes_point_extractor_r

/var/folders/2k/78nn7s4548986wsjh29rhj9w0000gn/T/ipykernel_31256/3361256823.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_panoptes_point_extractor_r[col] = df_panoptes_point_extractor_r[col].apply(lambda x: literal_eval(x) if pd.notnull(x) else [])
/var/folders/2k/78nn7s4548986wsjh29rhj9w0000gn/T/ipykernel_31256/3361256823.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_panoptes_point_extractor_r['x'] = df_panoptes_point_extractor_r[columns_keep_x].values.tolist()
/var/folders/2k/78nn7s454

,classification_id,user_name,user_id,workflow_id,task,created_at,subject_id,extractor,data.aggregation_version,data.frame0.T2_tool2_x,...,data.frame0.T2_tool3_x,data.frame0.T2_tool3_y,data.frame0.T2_tool1_x,data.frame0.T2_tool1_y,data.frame0.T4_tool3_x,data.frame0.T4_tool3_y,workflow_version,image_name,x,y
0,516391010,386fc0ec047b7e259744e72e8e64b9f9,ea57b1088a10fa7fef30ed0b344e2ca3,25351,T2,2023-10-16 10:10:44 UTC,92462758,point_extractor_by_frame,4.1.0,[509.775390625],...,NaN,NaN,[],[],NaN,NaN,51.133,Fer_FPE03-04-05_18122021_118_14.jpg,[509.775390625],[664.4249267578125]
2,520603679,3e5c95da2f84e6f04e971408dbdab8d7,1ed243d8863f93f280b5aecce0c2f80c,25351,T2,2023-11-05 18:25:57 UTC,92462758,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,[],[],NaN,NaN,51.133,Fer_FPE03-04-05_18122021_118_14.jpg,[486.3125],[679.75]
4,523311597,83dd7d12ba62f69e40d9bd8b5eeb23b0,dab35ca4005258caceaab79f37e23712,25351,T2,2023-11-18 02:33:13 UTC,92462758,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,[508.81048583984375],[665.7697143554688],NaN,NaN,51.133,Fer_FPE03-04-05_18122021_118_14.jpg,[508.81048583984375],[665.7697143554688]
6,528076882,5b7f6cd059096be78a76988c60f1c741,NaN,25351,T2,2023-12-09 19:12:49 UTC,92462758,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,[509.3125],[663.75],NaN,NaN,51.133,Fer_FPE03-04-05_18122021_118_14.jpg,[509.3125],[663.75]
8,529018098,386fa01a49140278f5494d0042b9f20a,27a60a267e4e0f1d976cac30038e8fe3,25351,T2,2023-12-14 04:32:06 UTC,92462758,point_extractor_by_frame,4.1.0,[509.70989990234375],...,NaN,NaN,[],[],NaN,NaN,51.133,Fer_FPE03-04-05_18122021_118_14.jpg,[509.70989990234375],[666.1571044921875]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404630,566921121,dce9e8bb730153bf0d0bad3f52b853e0,b0e10539db360fb487870c2b5dff8aaa,25351,T2,2024-06-11 15:39:30 UTC,92469395,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,[],[],NaN,NaN,51.133,Isa_ISVB01_27012023_57_29.jpg,[],[]
404632,567651844,940c519bb3437ec73a4fc6071eae5041,8e606b31ff94fd9d6ba2151bb5d1d030,25351,T2,2024-06-15 09:53:39 UTC,92469395,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,[],[],NaN,NaN,51.133,Isa_ISVB01_27012023_57_29.jpg,[],[]
404634,569258394,ba42128d23c5d1e18111062e13f8c2d3,00b4196b5d562fbd25d0585353c627f9,25351,T2,2024-06-24 12:16:33 UTC,92469395,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,[],[],NaN,NaN,51.133,Isa_ISVB01_27012023_57_29.jpg,[],[]
404636,569377745,17d2b0a53f67a0ccba57a9b0aec20f50,92c6091eaf50a7f1019c6cd81f8dd5e4,25351,T2,2024-06-24 23:40:00 UTC,92469395,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,[],[],NaN,NaN,51.133,Isa_ISVB01_27012023_57_29.jpg,[],[]


In [35]:
df_panoptes_point_extractor_r = df_panoptes_point_extractor_r[
    ['classification_id', 'user_name', 'user_id', 'workflow_id',  'workflow_version', 'task',
     'created_at', 'subject_id', "image_name",
     'x', 'y'
     ]].reset_index(drop=True)

df_panoptes_point_extractor_r

,classification_id,user_name,user_id,workflow_id,workflow_version,task,created_at,subject_id,image_name,x,y
0,516391010,386fc0ec047b7e259744e72e8e64b9f9,ea57b1088a10fa7fef30ed0b344e2ca3,25351,51.133,T2,2023-10-16 10:10:44 UTC,92462758,Fer_FPE03-04-05_18122021_118_14.jpg,[509.775390625],[664.4249267578125]
1,520603679,3e5c95da2f84e6f04e971408dbdab8d7,1ed243d8863f93f280b5aecce0c2f80c,25351,51.133,T2,2023-11-05 18:25:57 UTC,92462758,Fer_FPE03-04-05_18122021_118_14.jpg,[486.3125],[679.75]
2,523311597,83dd7d12ba62f69e40d9bd8b5eeb23b0,dab35ca4005258caceaab79f37e23712,25351,51.133,T2,2023-11-18 02:33:13 UTC,92462758,Fer_FPE03-04-05_18122021_118_14.jpg,[508.81048583984375],[665.7697143554688]
3,528076882,5b7f6cd059096be78a76988c60f1c741,NaN,25351,51.133,T2,2023-12-09 19:12:49 UTC,92462758,Fer_FPE03-04-05_18122021_118_14.jpg,[509.3125],[663.75]
4,529018098,386fa01a49140278f5494d0042b9f20a,27a60a267e4e0f1d976cac30038e8fe3,25351,51.133,T2,2023-12-14 04:32:06 UTC,92462758,Fer_FPE03-04-05_18122021_118_14.jpg,[509.70989990234375],[666.1571044921875]
...,...,...,...,...,...,...,...,...,...,...,...
202315,566921121,dce9e8bb730153bf0d0bad3f52b853e0,b0e10539db360fb487870c2b5dff8aaa,25351,51.133,T2,2024-06-11 15:39:30 UTC,92469395,Isa_ISVB01_27012023_57_29.jpg,[],[]
202316,567651844,940c519bb3437ec73a4fc6071eae5041,8e606b31ff94fd9d6ba2151bb5d1d030,25351,51.133,T2,2024-06-15 09:53:39 UTC,92469395,Isa_ISVB01_27012023_57_29.jpg,[],[]
202317,569258394,ba42128d23c5d1e18111062e13f8c2d3,00b4196b5d562fbd25d0585353c627f9,25351,51.133,T2,2024-06-24 12:16:33 UTC,92469395,Isa_ISVB01_27012023_57_29.jpg,[],[]
202318,569377745,17d2b0a53f67a0ccba57a9b0aec20f50,92c6091eaf50a7f1019c6cd81f8dd5e4,25351,51.133,T2,2024-06-24 23:40:00 UTC,92469395,Isa_ISVB01_27012023_57_29.jpg,[],[]


In [36]:
# explode the lists of marks per user into one row per mark
df_panoptes_point_extractor_r_ex = df_panoptes_point_extractor_r.apply(lambda x: x.explode() if x.name in ['x', 'y'] else x)

In [37]:
# images with no marks have NaN values in the 'merged_x' and 'merged_y' columns
df_panoptes_point_extractor_r_ex_dropped = df_panoptes_point_extractor_r_ex.dropna(subset=['x', 'y'], how='all').sort_values(by=['user_id', 'subject_id', 'task', 'created_at'])
df_panoptes_point_extractor_r_ex_dropped

,classification_id,user_name,user_id,workflow_id,workflow_version,task,created_at,subject_id,image_name,x,y
137313,525501702,2c71cd336686d99b2fddf9bf5c5e2d86,0013808b3d09505e4e150c365ee48a43,25351,51.133,T2,2023-11-29 02:20:34 UTC,92466390,Isa_ISCWN02_18012023_81_36.jpg,335.898926,655.554871
199593,537681646,10994132e025a2d3bee8971688da1fe5,003883280e2befdf34edfb257bd8a791,25351,51.133,T2,2024-01-26 16:15:54 UTC,92461153,Fer_FPE01-06_18122021_32_33.jpg,16.868912,245.235474
87112,537682677,10994132e025a2d3bee8971688da1fe5,003883280e2befdf34edfb257bd8a791,25351,51.133,T2,2024-01-26 16:20:06 UTC,92463273,Fer_FPM01-02_20012023_26_108.jpg,652.825562,596.568848
87112,537682677,10994132e025a2d3bee8971688da1fe5,003883280e2befdf34edfb257bd8a791,25351,51.133,T2,2024-01-26 16:20:06 UTC,92463273,Fer_FPM01-02_20012023_26_108.jpg,233.017883,561.024475
111210,537681792,10994132e025a2d3bee8971688da1fe5,003883280e2befdf34edfb257bd8a791,25351,51.133,T2,2024-01-26 16:16:38 UTC,92465107,Fer_FPM01-02_20012023_52_74.jpg,67.784637,594.647522
...,...,...,...,...,...,...,...,...,...,...,...
44075,563671982,ff34a702faa50d4664aaa32e1f4aa569,NaN,25351,51.133,T2,2024-05-24 17:24:30 UTC,92469726,Isa_ISVB01_27012023_75_37.jpg,349.43692,122.996864
76393,518717765,9b7e5901c61869dc6c15ef440db39903,NaN,25351,51.133,T2,2023-10-26 22:26:33 UTC,92469727,Isa_ISVB02_27012023_154.jpg,456.804688,129.300003
73028,518469002,1d23a3d52d2335e30eda5f7fd35beb05,NaN,25351,51.133,T2,2023-10-26 01:01:00 UTC,92469733,Isa_ISVB01_27012023_75_36.jpg,582.7146,657.208252
73035,535441276,b6254211ad7527a1e3d29e0d8ba29660,NaN,25351,51.133,T2,2024-01-18 22:56:05 UTC,92469733,Isa_ISVB01_27012023_75_36.jpg,599.316406,675.324341


In [38]:
# cast x and y to int
df_panoptes_point_extractor_r_ex_dropped = df_panoptes_point_extractor_r_ex_dropped.astype({'x': 'int32', 'y': 'int32'})
df_panoptes_point_extractor_r_ex_dropped

,classification_id,user_name,user_id,workflow_id,workflow_version,task,created_at,subject_id,image_name,x,y
137313,525501702,2c71cd336686d99b2fddf9bf5c5e2d86,0013808b3d09505e4e150c365ee48a43,25351,51.133,T2,2023-11-29 02:20:34 UTC,92466390,Isa_ISCWN02_18012023_81_36.jpg,335,655
199593,537681646,10994132e025a2d3bee8971688da1fe5,003883280e2befdf34edfb257bd8a791,25351,51.133,T2,2024-01-26 16:15:54 UTC,92461153,Fer_FPE01-06_18122021_32_33.jpg,16,245
87112,537682677,10994132e025a2d3bee8971688da1fe5,003883280e2befdf34edfb257bd8a791,25351,51.133,T2,2024-01-26 16:20:06 UTC,92463273,Fer_FPM01-02_20012023_26_108.jpg,652,596
87112,537682677,10994132e025a2d3bee8971688da1fe5,003883280e2befdf34edfb257bd8a791,25351,51.133,T2,2024-01-26 16:20:06 UTC,92463273,Fer_FPM01-02_20012023_26_108.jpg,233,561
111210,537681792,10994132e025a2d3bee8971688da1fe5,003883280e2befdf34edfb257bd8a791,25351,51.133,T2,2024-01-26 16:16:38 UTC,92465107,Fer_FPM01-02_20012023_52_74.jpg,67,594
...,...,...,...,...,...,...,...,...,...,...,...
44075,563671982,ff34a702faa50d4664aaa32e1f4aa569,NaN,25351,51.133,T2,2024-05-24 17:24:30 UTC,92469726,Isa_ISVB01_27012023_75_37.jpg,349,122
76393,518717765,9b7e5901c61869dc6c15ef440db39903,NaN,25351,51.133,T2,2023-10-26 22:26:33 UTC,92469727,Isa_ISVB02_27012023_154.jpg,456,129
73028,518469002,1d23a3d52d2335e30eda5f7fd35beb05,NaN,25351,51.133,T2,2023-10-26 01:01:00 UTC,92469733,Isa_ISVB01_27012023_75_36.jpg,582,657
73035,535441276,b6254211ad7527a1e3d29e0d8ba29660,NaN,25351,51.133,T2,2024-01-18 22:56:05 UTC,92469733,Isa_ISVB01_27012023_75_36.jpg,599,675


In [39]:
print(f"write file to: {config['flat_panoptes_points']}")
df_panoptes_point_extractor_r_ex_dropped.to_csv(config["flat_panoptes_points"], sep=",", index = False)

write file to: /Users/christian/data/zooniverse/2024_09_25_analysis/Iguanas 4th launch/flat_panoptes_points_Iguanas 4th launch.csv


## Inspecting the results
Check the numbers for a single subject_id

In [ ]:
### Looks the images in question

subject_id_2 = 72373250 
df_debug = df_panoptes_point_extractor_r_ex_dropped[(df_panoptes_point_extractor_r_ex_dropped.subject_id == subject_id_2)]
df_debug

In [ ]:
df_debug.groupby('user_name').size()


In [ ]:
df_debug[df_debug.user_name == "CallieSanDiego"]

## Download images
iguanas-from-above-subjects_with_url.csv will be used to track which url was already downlaoded.

In [ ]:
## save the file the extra columns we need for downloading.
df_subjects.to_csv(output_path / "iguanas-from-above-subjects_with_url.csv")


# read the modified csv
df_subjects = pd.read_csv(output_path / "iguanas-from-above-subjects_with_url.csv")


In [ ]:
# df_subjects = pd.read_csv(output_path / "iguanas-from-above-subjects_with_url.csv")

# downoaded_images_path = Path("./data/downloaded_images")
# downoaded_images_path.mkdir(exist_ok=True, parents=True)
# return_val = True
# # df = df_subjects[df_subjects.subject_id.isin([44660616, 47968406])]
# # df = df_subjects[df_subjects.subject_id.isin([44660616, 47968406])]
# for index, row in df_subjects[df_subjects.workflow_id.isin([workflow_id_p1])].iterrows():
#     # Only download if necessary
#     if pd.isna(row.get("filepath")) or not row.get("filepath", False):
#         flight_code = row['flight_code']
#         url = row['url']
#         image_name = Path(row['image_name']).name
#         # Extract the filename from the URL and create a unique name using index
#         filename = downoaded_images_path.joinpath(f"{image_name}_{row['subject_id']}_{flight_code}.jpeg")
#         df_subjects.loc[index, 'filepath'] = filename
#         # Download the image
#         return_val = download_image(url, filename)
# 
#         # print(f"Downloaded {filename}")
#     if return_val == False:
#         print("there was a problem")
#         # break
        

In [ ]:
df_subjects.to_csv(output_path / "iguanas-from-above-subjects_with_url.csv")